# Survival Analysis: From Kaplan–Meier to Cox Regression

## Learning objectives
By the end of this notebook, you will be able to:

1. explain time-to-event outcomes and right censoring;
2. estimate and interpret a Kaplan–Meier survival curve with confidence intervals;
3. compare groups with a log-rank test;
4. fit and interpret a Cox proportional hazards model; and
5. check key assumptions without overstating causal conclusions.

## Core ideas
Survival analysis models the time $T$ until an event such as churn, purchase, failure, recovery, or death. If observation ends before the event occurs, the record is **right-censored**: we know only that $T$ exceeds the observed duration.

The survival function is

$$S(t)=P(T>t),$$

and the hazard $h(t)$ is the instantaneous event rate among subjects still at risk. The Kaplan–Meier estimator is

$$\widehat{S}(t)=\prod_{t_i\le t}\left(1-\frac{d_i}{n_i}\right),$$

where $d_i$ is the number of events at time $t_i$ and $n_i$ is the risk set immediately before that time.

> Censoring is not an event. Standard methods assume censoring is non-informative conditional on the variables in the analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import Markdown, display
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.datasets import load_lung
from lifelines.statistics import logrank_test, proportional_hazard_test

pd.set_option("display.max_columns", 20)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Maintained example data bundled with lifelines; no local CSV is required.
data = load_lung().copy()

# lifelines releases use either {0, 1} or the original R coding {1, 2}.
status_values = set(data["status"].dropna().astype(int).unique())
if status_values <= {0, 1}:
    data["event"] = data["status"].astype(int)
elif status_values <= {1, 2}:
    data["event"] = data["status"].eq(2).astype(int)
else:
    raise ValueError(f"Unexpected status coding: {sorted(status_values)}")

data["sex_label"] = data["sex"].map({1: "Male", 2: "Female"})
data.head()

## 1. Data and event definition

The lung-cancer study contains 228 patients:

- `time`: observed duration in days;
- `event`: 1 if death was observed, 0 if right-censored;
- `age`: age in years;
- `sex`: 1 = male, 2 = female in the source data;
- `ph.ecog`: physician-rated ECOG performance score (higher is worse);
- `ph.karno` / `pat.karno`: physician- and patient-rated Karnofsky scores (higher is better);
- `meal.cal`: calories consumed at meals;
- `wt.loss`: weight loss over the previous six months.

The clinical dataset is used to teach methods. It does not establish that a characteristic causes longer or shorter survival.

In [ ]:
audit = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing": data.isna().sum(),
    "missing_pct": data.isna().mean().mul(100).round(1),
})
audit

In [ ]:
summary = pd.Series({
    "patients": len(data),
    "observed_events": int(data["event"].sum()),
    "right_censored": int((1 - data["event"]).sum()),
    "event_fraction": data["event"].mean(),
    "minimum_follow_up_days": data["time"].min(),
    "maximum_follow_up_days": data["time"].max(),
})
summary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = data["sex_label"].value_counts()
counts.plot.bar(ax=ax, color=["#457b9d", "#e76f51"])
ax.bar_label(ax.containers[0])
ax.set(title="Patients by sex", xlabel="", ylabel="Patients")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

Counts describe the sample composition only. They do not show survival differences because follow-up time and censoring must also be considered.

In [ ]:
event_table_by_sex = pd.crosstab(
    data["sex_label"],
    data["event"].map({0: "Right-censored", 1: "Event observed"}),
)
event_table_by_sex

Raw event counts are not event rates. A group can accumulate more observed events because it is larger or followed longer. Kaplan–Meier estimation accounts for unequal observed durations and right censoring.

In [ ]:
data.groupby("sex_label").agg(
    patients=("time", "size"),
    observed_events=("event", "sum"),
    median_observed_time=("time", "median"),
    mean_age=("age", "mean"),
).round(1)

The median of the **observed durations** is not automatically the median survival time: censored records prevent direct interpretation. Estimate median survival from the Kaplan–Meier curve instead.

In [ ]:
display(Markdown(r"""## 2. Kaplan–Meier estimate

`event_observed` must be `1` for an observed event and `0` for a right-censored record. The fitted curve is a step function; Greenwood-based confidence intervals quantify sampling uncertainty under the estimator's assumptions."""))

In [ ]:
kmf = KaplanMeierFitter(label="All patients")
kmf.fit(durations=data["time"], event_observed=data["event"])

In [ ]:
time_points = pd.Series([30, 90, 180, 365], name="day")
pd.DataFrame({
    "day": time_points,
    "survival_probability": kmf.predict(time_points).to_numpy(),
}).style.format({"survival_probability": "{:.1%}"})

In [ ]:
print(f"Observed follow-up: {data['time'].min():.0f} to {data['time'].max():.0f} days")
print(f"Kaplan–Meier median survival: {kmf.median_survival_time_:.0f} days")

The median survival is the first time at which $\widehat{S}(t)\le 0.5$. It is not the average time to death, and it can be undefined when the estimated curve never falls to 0.5.

In [ ]:
event_table = kmf.event_table
event_table.head(10)

In [ ]:
first_steps = event_table.loc[event_table.index[:3], ["at_risk", "observed", "censored"]].copy()
first_steps["conditional_survival"] = (
    first_steps["at_risk"] - first_steps["observed"]
) / first_steps["at_risk"]
first_steps["km_product"] = first_steps["conditional_survival"].cumprod()
first_steps

In [ ]:
np.testing.assert_allclose(
    first_steps["km_product"].iloc[-1],
    kmf.predict(first_steps.index[-1]),
)
print("Manual product matches lifelines at the final displayed time.")

### Reading the event table

- `at_risk`: subjects under observation and event-free immediately before the time;
- `observed`: events at that time;
- `censored`: subjects whose follow-up ends without an observed event;
- `removed`: `observed + censored`;
- `entrance`: subjects entering the risk set.

At an event time, conditional survival is $(n_i-d_i)/n_i$. The Kaplan–Meier estimate multiplies those conditional probabilities through time. Censored records reduce later risk sets but are **not** counted as events.

The estimator can be written as:

$$\hat{S}(t) = \prod_{t_i \lt t} \frac{n_i - d_i}{n_i}$$

where  $d_i$  are the number of death events at time  $t$  and  $n_i$  is the number of subjects at risk of death just prior to time  $t$. 

What the above essentially tells us is that the value of the survival function for time $t$, is the product of the survival probabilities for all individual time periods leading up to time $t$. 

We can define the survival probability for an individual time period as follows:

$$S_t = \frac{\substack{\text{Number of subjects} \\ \text{at risk at the start}} - \substack{\text{Number of subjects} \\ \text{that died}}}{\substack{\text{Number of subjects} \\ \text{at risk at the start}}}$$

**NOTE** the number of deaths in the above formula does not include the number of censored observations.  

<u>Get information about the time 0

In [ ]:
event_at_0 = kmf.event_table.iloc[0,:]
event_at_0

<u> Get the survival probability at time 0

In [ ]:
#calculate the survival probability for t=0
surv_for_0 = (event_at_0.at_risk - event_at_0.observed)/event_at_0.at_risk
surv_for_0

<u>Get information about the time 5

In [ ]:
event_at_5 = kmf.event_table.iloc[1,:]
event_at_5

<u>Get the survival probability at time 5

In [ ]:
surv_for_5 = (event_at_5.at_risk - event_at_5.observed)/event_at_5.at_risk
surv_for_5


<u>Get information about the time 11

In [ ]:
event_at_11 = kmf.event_table.iloc[2,:]
event_at_11

In [ ]:
surv_for_11 = (event_at_11.at_risk - event_at_11.observed)/event_at_11.at_risk
surv_for_11


The factor at day 11 is a **conditional** survival probability: survival through that event time given survival immediately beforehand. The cumulative estimate multiplies all prior event-time factors:

$$\widehat{S}(11)=\frac{228}{228}\times\frac{227}{228}\times\frac{224}{227}\approx0.982.$$

The exact risk-set values are data-specific; the code above derives them rather than hard-coding them.

In [ ]:
# The probability to survive after time 5 is:
surv_after_5 = surv_for_0 * surv_for_5
surv_after_5

In [ ]:
# The probability to survive after time 11 is:
surv_after_11 = surv_for_0 * surv_for_5 * surv_for_11
surv_after_11

#kmf.predict(11) # calculates the same

### Survival and cumulative-incidence estimates

`predict(t)` returns the estimated survival probability at time $t$. This is an estimate, not an individual-level prediction.

`survival_function_` stores $\widehat{S}(t)$ at observed timeline values. With a single event type, `cumulative_density_` is $1-\widehat{S}(t)$.

In [ ]:
kmf.survival_function_

In [ ]:
# Probabaility of die at time t, calculate cumulative density, which is equal
# 1 - KM_estimate

kmf.cumulative_density_

#kmf.cumulative_density_.head()

In [ ]:
kmf.plot_cumulative_density()

In [ ]:
# Get confidence interval
kmf.confidence_interval_

### Kaplan–Meier curve with confidence interval and risk table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
kmf.plot_survival_function(ax=ax, ci_show=True, show_censors=True)
ax.axhline(0.5, color="#666666", linestyle=":", linewidth=1)
ax.axvline(kmf.median_survival_time_, color="#e76f51", linestyle="--", linewidth=1)
ax.set(
    title="Kaplan–Meier estimate: all patients",
    xlabel="Days",
    ylabel="Estimated survival probability",
    ylim=(0, 1.02),
)
plt.tight_layout()
plt.show()

risk_times = [0, 250, 500, 750, 1000]
risk_table = kmf.event_table["at_risk"].reindex(risk_times, method="ffill")
pd.DataFrame({"day": risk_times, "patients_at_risk": risk_table.to_numpy()})

Vertical tick marks indicate censored observations. The at-risk table prevents a common mistake: over-interpreting the unstable tail of a curve after few subjects remain.

In [ ]:
display(Markdown(r"""### Interpreting the curve

The curve estimates the survival experience of this sample under independent censoring. Confidence bands widen as the risk set shrinks. Do not read the curve as an individual's prognosis, and do not compare groups visually without reporting uncertainty and a justified statistical test."""))

In [ ]:
pd.DataFrame({
    "estimate": [kmf.median_survival_time_],
    "unit": ["days"],
}, index=["Kaplan–Meier median survival"])

The estimated median is the time by which the estimated survival probability reaches 50%. It is a quantile, not an arithmetic mean.

In [ ]:
# Get the median survival days

kmf.median_survival_time_

## 3. Compare groups with Kaplan–Meier curves and a log-rank test

The log-rank test compares observed and expected events across the full follow-up period. It is most sensitive when hazards are roughly proportional. A small p-value indicates evidence that the survival functions differ; it does not measure effect size or establish causality.

- $H_0$: the groups have the same survival function.
- $H_1$: the survival functions differ at one or more times.

Choose the significance level before inspecting results and report the test statistic, p-value, curves, risk sets, and an effect estimate.

The test aggregates $(O-E)$ contributions over event times and scales them by their variance. Censored observations affect risk sets but do not count as events.

For two groups, the statistic is asymptotically chi-squared with one degree of freedom:

$$\chi^2=\frac{\left[\sum_t(O_{1t}-E_{1t})\right]^2}{\sum_t V_{1t}}.$$

Here $O_{1t}$ and $E_{1t}$ are observed and expected group-1 events at time $t$, and $V_{1t}$ is the corresponding variance contribution.

In [ ]:
male = data.loc[data["sex_label"].eq("Male")].copy()
female = data.loc[data["sex_label"].eq("Female")].copy()

print(f"Male: {len(male)} patients; Female: {len(female)} patients")

In [ ]:
kmf_m = KaplanMeierFitter(label="Male").fit(male["time"], male["event"])
kmf_f = KaplanMeierFitter(label="Female").fit(female["time"], female["event"])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
kmf_m.plot_survival_function(ax=ax, ci_show=True, show_censors=True)
kmf_f.plot_survival_function(ax=ax, ci_show=True, show_censors=True)
ax.set(
    title="Kaplan–Meier survival by sex",
    xlabel="Days",
    ylabel="Estimated survival probability",
    ylim=(0, 1.02),
)
plt.tight_layout()
plt.show()

group_risk_table = pd.DataFrame({
    "day": risk_times,
    "male_at_risk": kmf_m.event_table["at_risk"].reindex(risk_times, method="ffill").to_numpy(),
    "female_at_risk": kmf_f.event_table["at_risk"].reindex(risk_times, method="ffill").to_numpy(),
})
group_risk_table

In [ ]:
# Get the event table for males
kmf_m.event_table

In [ ]:
# Get the event table for females
kmf_f.event_table

The event tables show how group-specific risk sets change over time. Differences between sample curves should be summarized with uncertainty and a formal comparison, not inferred from event counts alone.

In [ ]:
pd.DataFrame({
    "group": ["Male", "Female"],
    "patients": [len(male), len(female)],
    "events": [int(male["event"].sum()), int(female["event"].sum())],
    "median_survival_days": [kmf_m.median_survival_time_, kmf_f.median_survival_time_],
})

In [ ]:
logrank = logrank_test(
    male["time"],
    female["time"],
    event_observed_A=male["event"],
    event_observed_B=female["event"],
)
pd.Series({
    "test_statistic": logrank.test_statistic,
    "p_value": logrank.p_value,
}).round(4)

In [ ]:
display(Markdown(r"""At $\alpha=0.05$, reject $H_0$ only when the p-value is below 0.05. Failure to reject does **not** prove equality; it may reflect limited power. The unadjusted test also does not control for age, performance status, or other covariates."""))

In [ ]:
alpha = 0.05
if logrank.p_value < alpha:
    conclusion = "Evidence of a difference between the group survival functions."
else:
    conclusion = "Insufficient evidence of a difference between the group survival functions."
print(f"p = {logrank.p_value:.4g}. {conclusion}")

In [ ]:
display(Markdown(r"""### Limitations of the group comparison

The log-rank result is unadjusted and relies on independent observations and non-informative censoring. Crossing curves or strongly non-proportional hazards can make a single log-rank test or hazard ratio misleading. Consider time-varying effects or restricted mean survival time when proportional hazards is implausible."""))

The next section uses Cox regression to estimate covariate-adjusted hazard ratios. Adjustment can reduce confounding by measured covariates, but observational adjustment does not by itself identify causal effects.

Report exact p-values and effect estimates rather than only labels such as “significant.” Statistical significance does not imply clinical importance.

## 4. Cox proportional hazards model

The Cox model relates covariates to the hazard while leaving the baseline hazard unspecified:

$$h(t\mid\mathbf{x})=h_0(t)\exp(\mathbf{x}^{\mathsf T}\boldsymbol\beta).$$

Exponentiated coefficients are **hazard ratios (HRs)**. An HR above 1 indicates a higher instantaneous event rate, an HR below 1 indicates a lower rate, and an HR of 1 indicates no multiplicative difference—conditional on the included covariates.

The baseline hazard $h_0(t)$ may vary freely over time. The proportional-hazards assumption requires each covariate's hazard ratio to remain constant over time unless time-varying effects are modeled explicitly.

### Main assumptions

1. **Proportional hazards:** each modeled hazard ratio is constant over time.
2. **Independent observations:** dependence or clustering must be modeled or handled with robust uncertainty.
3. **Appropriate functional form:** continuous covariates are linear on the log-hazard scale unless transformed.
4. **Non-informative censoring:** conditional on modeled information, censoring is unrelated to future event risk.
5. **No severe multicollinearity or influential data errors.**

The hazard is not the slope of the survival curve. It is a conditional instantaneous rate; specifically, $h(t)=-S'(t)/S(t)$ when the derivative exists.

We use a compact teaching model with age, sex, ECOG performance status, physician-rated Karnofsky score, and weight loss. Scaling age, Karnofsky score, and weight loss by 10 makes the hazard ratios easier to interpret. Complete-case analysis is convenient for teaching but can be biased when missingness is systematic.

In [ ]:
cox_data = data[["time", "event", "age", "sex", "ph.ecog", "ph.karno", "wt.loss"]].copy()
cox_data["male"] = cox_data["sex"].eq(1).astype(int)
cox_data["age_per_10y"] = cox_data["age"] / 10
cox_data["karno_per_10"] = cox_data["ph.karno"] / 10
cox_data["weight_loss_per_10lb"] = cox_data["wt.loss"] / 10
cox_data = cox_data.drop(columns=["sex", "age", "ph.karno"]).dropna()

print(f"Complete cases: {len(cox_data)} of {len(data)} patients")
cox_data.head()

In [ ]:
cox_data.describe().T[["count", "mean", "std", "min", "max"]].round(2)

In [ ]:
display(Markdown(r"""The binary variable `male` compares male with female patients. Continuous-variable HRs correspond to the displayed increments: 10 years of age, 10 Karnofsky points, and 10 pounds of weight loss."""))

In [ ]:
cph = CoxPHFitter(penalizer=0.01)
cph.fit(cox_data, duration_col="time", event_col="event")
cph.print_summary(decimals=3)

hazard_ratios = cph.summary[[
    "exp(coef)",
    "exp(coef) lower 95%",
    "exp(coef) upper 95%",
    "p",
]].rename(columns={
    "exp(coef)": "hazard_ratio",
    "exp(coef) lower 95%": "ci_low",
    "exp(coef) upper 95%": "ci_high",
    "p": "p_value",
})
hazard_ratios.round(3)

### Interpreting hazard ratios

For a one-unit increase in a model covariate, $HR=\exp(\beta)$ multiplies the hazard while other included covariates are held fixed. For example, HR = 1.20 means a 20% higher instantaneous event rate—not 20% shorter survival and not a 20 percentage-point increase in event probability.

A 95% confidence interval containing 1 indicates that the data are compatible with no multiplicative association at the 5% level. Report the full interval, not only whether it crosses 1.

The model output above reports adjusted hazard ratios, 95% confidence intervals, and p-values. Interpret the direction, magnitude, and uncertainty together; avoid reducing results to a binary significance label.

Interpret the values produced by the fitted model rather than copying fixed claims into a report. For example, the `male` row compares male with female patients conditional on the other modeled covariates; `age_per_10y` represents a 10-year increase. Associations may still reflect residual confounding, measurement error, selection, and missing data.

### Check proportional hazards

The Schoenfeld-residual test assesses whether each coefficient appears to vary with time. A small p-value flags potential non-proportionality; it is a diagnostic, not proof. Also inspect plots and use subject-matter knowledge.

In [ ]:
ph_test = proportional_hazard_test(cph, cox_data, time_transform="rank")
ph_test.summary[["test_statistic", "p"]].sort_values("p").round(3)

In [ ]:
display(Markdown(r"""## Key takeaways

- Preserve censoring information; dropping censored records biases time-to-event analysis.
- Use Kaplan–Meier curves with confidence intervals and risk tables for unadjusted summaries.
- A log-rank p-value tests a difference but does not measure its size or prove causality.
- Interpret Cox coefficients as hazard ratios, not probability ratios or changes in survival time.
- Check proportional hazards, functional form, missingness, dependence, and influential observations before reporting results.

### Practice
1. Add a Kaplan–Meier comparison by ECOG score.
2. Plot Cox coefficient confidence intervals.
3. Replace complete-case analysis with a justified missing-data strategy.
4. Investigate a flagged proportional-hazards violation with a time-varying effect."""))